# Modelos Estatísticos de Previsão da Inflação

Este notebook (NB8) dá o próximo passo do projeto:

- Usar **modelos estatísticos clássicos** para prever inflação
- Integrar:
  - IPCA e núcleos
  - SELIC e juros reais
  - Blocos macroeconômicos
  - Curva de Phillips (hiato / inércia)

Foco:
- Previsão de IPCA mensal (curto prazo)
- Previsão de IPCA 12m à frente (médio prazo)

Compatível com Jupytext: `08.modelos_estatisticos_previsao.py`.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
import statsmodels.api as sm

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

print("PYTHONPATH atualizado:", ROOT)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Carregar datasets processados

Usamos:
- `ipca_selic_features.parquet` (NB5)
- `ipca_blocos_macro.parquet` (NB6)
- `curva_phillips_features.parquet` (NB7)

In [ ]:
path_selic = os.path.join(ROOT, "data", "processed", "ipca_selic_features.parquet")
path_blocos = os.path.join(ROOT, "data", "processed", "ipca_blocos_macro.parquet")
path_phillips = os.path.join(ROOT, "data", "processed", "curva_phillips_features.parquet")

df_selic = pd.read_parquet(path_selic)
df_blocos = pd.read_parquet(path_blocos)
df_phillips = pd.read_parquet(path_phillips)

df_selic.index = pd.to_datetime(df_selic.index)
df_blocos.index = pd.to_datetime(df_blocos.index)
df_phillips.index = pd.to_datetime(df_phillips.index)

df_selic.tail(), df_blocos.tail(), df_phillips.tail()

## 2. Base consolidada para previsão

Vamos montar uma base única com:
- IPCA (target)
- SELIC e derivados
- Blocos macroeconômicos
- Variáveis da Curva de Phillips

In [ ]:
df_all = df_selic.join(df_blocos, how="inner").join(df_phillips, how="inner")
df_all = df_all.sort_index()
df_all.tail()

## 3. Definir targets de previsão

Vamos trabalhar com dois alvos:

1. **IPCA mensal (t+1)** – previsão de curto prazo  
2. **IPCA 12m à frente (t+12)** – previsão de médio prazo

In [ ]:
# IPCA mensal (já está em df_selic["ipca"])
df_all["ipca_t"] = df_all["ipca"]

# IPCA 12m acumulado (já existe como ipca_12m em df_selic)
df_all["ipca_12m_t"] = df_all["ipca_12m"]

# Targets futuros
df_all["ipca_t_plus_1"] = df_all["ipca_t"].shift(-1)
df_all["ipca_12m_t_plus_12"] = df_all["ipca_12m_t"].shift(-12)

df_all.tail(15)

## 4. Separar treino e teste

Usamos uma divisão temporal:
- treino: até uma certa data
- teste: últimos N meses

In [ ]:
split_date = df_all.index[-36]  # últimos 36 meses para teste

df_train = df_all[df_all.index <= split_date].copy()
df_test = df_all[df_all.index > split_date].copy()

print("Treino até:", split_date.date())
print("Tamanho treino:", len(df_train))
print("Tamanho teste:", len(df_test))

## 5. Modelo ARIMA univariado (baseline)

Primeiro, um modelo simples:
- ARIMA apenas com IPCA mensal
- Serve como benchmark

In [ ]:
train_ipca = df_train["ipca_t"].dropna()
test_ipca = df_test["ipca_t"].dropna()

arima_order = (1, 0, 1)  # pode ser ajustado depois com auto-arima, AIC, etc.

model_arima = ARIMA(train_ipca, order=arima_order)
res_arima = model_arima.fit()
print(res_arima.summary())

In [ ]:
# Previsão para o período de teste
forecast_arima = res_arima.forecast(steps=len(test_ipca))
rmse_arima = mean_squared_error(test_ipca, forecast_arima, squared=False)

print("RMSE ARIMA (IPCA t+1):", rmse_arima)

plt.figure(figsize=(10, 4))
plt.plot(train_ipca.index, train_ipca, label="Treino")
plt.plot(test_ipca.index, test_ipca, label="Teste - Observado", color="black")
plt.plot(test_ipca.index, forecast_arima, label="Teste - ARIMA", color="darkorange")
plt.title("ARIMA – Previsão de IPCA mensal (t+1)")
plt.xlabel("Data")
plt.ylabel("IPCA (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Modelo ARIMAX (com variáveis explicativas)

Agora incluímos:
- SELIC
- SELIC real
- hiato_proxy (Curva de Phillips)
- alguns blocos macroeconômicos

para prever **IPCA mensal (t+1)**.

In [ ]:
# Seleção de features explicativas
features_exog = []

# SELIC e derivados
features_exog += ["selic", "selic_real", "ipca_momentum_3m", "selic_minus_momentum"]

# Hiato proxy (NB7)
if "hiato_proxy" in df_all.columns:
    features_exog.append("hiato_proxy")

# Alguns blocos macro (se existirem)
features_exog += [c for c in df_all.columns if c.startswith("bloco_") and "12m" not in c]

features_exog = list(dict.fromkeys(features_exog))  # remover duplicatas
features_exog

In [ ]:
train_exog = df_train[features_exog].dropna()
test_exog = df_test[features_exog].dropna()

# alinhar target com exog
common_index_train = train_ipca.index.intersection(train_exog.index)
common_index_test = test_ipca.index.intersection(test_exog.index)

train_ipca_aligned = train_ipca.loc[common_index_train]
train_exog_aligned = train_exog.loc[common_index_train]

test_ipca_aligned = test_ipca.loc[common_index_test]
test_exog_aligned = test_exog.loc[common_index_test]

In [ ]:
model_arimax = ARIMA(train_ipca_aligned, order=arima_order, exog=train_exog_aligned)
res_arimax = model_arimax.fit()
print(res_arimax.summary())

In [ ]:
forecast_arimax = res_arimax.forecast(steps=len(test_ipca_aligned), exog=test_exog_aligned)
rmse_arimax = mean_squared_error(test_ipca_aligned, forecast_arimax, squared=False)

print("RMSE ARIMAX (IPCA t+1):", rmse_arimax)

plt.figure(figsize=(10, 4))
plt.plot(train_ipca_aligned.index, train_ipca_aligned, label="Treino")
plt.plot(test_ipca_aligned.index, test_ipca_aligned, label="Teste - Observado", color="black")
plt.plot(test_ipca_aligned.index, forecast_arimax, label="Teste - ARIMAX", color="darkgreen")
plt.title("ARIMAX – Previsão de IPCA mensal (t+1)")
plt.xlabel("Data")
plt.ylabel("IPCA (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Previsão de IPCA 12m à frente (médio prazo)

Agora usamos:
- target: `ipca_12m_t_plus_12`
- features: mesmas do ARIMAX

Modelo: regressão linear simples (baseline de médio prazo).

In [ ]:
df_med = df_all.copy()

cols_med = features_exog + ["ipca_12m_t_plus_12"]
df_med = df_med[cols_med].dropna()

split_date_med = df_med.index[-36]

df_med_train = df_med[df_med.index <= split_date_med].copy()
df_med_test = df_med[df_med.index > split_date_med].copy()

y_train_med = df_med_train["ipca_12m_t_plus_12"]
X_train_med = df_med_train[features_exog]
X_train_med = sm.add_constant(X_train_med)

y_test_med = df_med_test["ipca_12m_t_plus_12"]
X_test_med = df_med_test[features_exog]
X_test_med = sm.add_constant(X_test_med)

model_med = sm.OLS(y_train_med, X_train_med)
res_med = model_med.fit()
print(res_med.summary())

In [ ]:
y_pred_med = res_med.predict(X_test_med)
rmse_med = mean_squared_error(y_test_med, y_pred_med, squared=False)

print("RMSE Regressão (IPCA 12m à frente):", rmse_med)

plt.figure(figsize=(10, 4))
plt.plot(y_test_med.index, y_test_med, label="Observado", color="black")
plt.plot(y_test_med.index, y_pred_med, label="Previsto", color="darkred")
plt.title("Regressão – Previsão de IPCA 12m à frente")
plt.xlabel("Data")
plt.ylabel("IPCA 12m (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Dataset final de avaliação e comparação

Vamos salvar:
- previsões ARIMA e ARIMAX para IPCA t+1
- previsões de regressão para IPCA 12m à frente

Isso será útil no NB9 (modelos de ML) para comparar desempenho.

In [ ]:
df_eval = pd.DataFrame(
    {
        "ipca_t_test": test_ipca_aligned,
        "arima_ipca_t_plus_1": forecast_arima.reindex(test_ipca_aligned.index),
        "arimax_ipca_t_plus_1": forecast_arimax,
    }
)

df_eval_12m = pd.DataFrame(
    {
        "ipca_12m_t_plus_12_test": y_test_med,
        "reg_ipca_12m_t_plus_12": y_pred_med,
    }
)

df_eval.tail(), df_eval_12m.tail()

In [ ]:
output_dir = os.path.join(ROOT, "data", "processed")
os.makedirs(output_dir, exist_ok=True)

path_eval_short = os.path.join(output_dir, "eval_estatistico_ipca_t_plus_1.parquet")
path_eval_med = os.path.join(output_dir, "eval_estatistico_ipca_12m_ahead.parquet")

df_eval.to_parquet(path_eval_short)
df_eval_12m.to_parquet(path_eval_med)

print("Salvo:", path_eval_short)
print("Salvo:", path_eval_med)

## 9. Conclusões e próximos passos

- O ARIMA univariado serve como **baseline**.
- O ARIMAX mostra o ganho de incluir:
  - SELIC
  - SELIC real
  - hiato proxy
  - blocos macroeconômicos
- A regressão para IPCA 12m à frente é um primeiro passo para cenários de médio prazo.

No **NB9 (Modelos de Machine Learning)**, vamos:
- usar os mesmos targets
- usar as mesmas features (e mais algumas)
- comparar:
  - ARIMA / ARIMAX / Regressão
  - Random Forest / XGBoost / outros